<a href="https://colab.research.google.com/github/Darioldeveloper-dev/CSC4110-G01/blob/main/G01_U9_3_publishing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 4110 Machine Learning
## Group G01: Student Performance Factors
### Task 9.3: Publishing - GitHub, Smoke Test and Online App

- **Official task owner:** Whole group (each member writes at least one part and makes at least one GitHub commit)
- **Module:** 10

**Contents**
1. Setup
2. README.md
3. Smoke test
4. Deployment checklist
5. Written answer

In [1]:
# Mount Drive and set the project path
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
base = Path("/content/drive/MyDrive/CSC4110_G01")
print(sorted(p.name for p in base.iterdir()))

Mounted at /content/drive
['app', 'data', 'models', 'notebooks', 'outputs', 'report', 'tests']


In [2]:
# Part (a): write README.md with run instructions, the public app link, the owners table and the principal paragraph
readme = """# CSC 4110 Group G01: Student Performance Factors

Kibabii University, CSC 4110 Machine Learning. We built and tested two models for Elimu Bora Trust on the public Student Performance Factors dataset: one predicts a student's exam score, the other suggests which students to invite for a mentorship scholarship interview (score of 70 or more).

## Public app
**https://REPLACE-WITH-YOUR-APP-NAME.streamlit.app**  (replace after deploying on Streamlit Community Cloud)

## Who owned each task
| Task | Content | Owner |
|---|---|---|
| 1 | Set-up and data | Whole group |
| 2.1 | Data inspection and cleaning | Dariol Temko (COM/0048/23) |
| 2.2 | Transformation, EDA and split | Raphael Wanyama (COM/0019/23) |
| 3.1 | Linear regression and gradient descent | Elisha Eddie (COM/0066/23) |
| 3.2 | Polynomial and regularised regression | Abdul Razak (COM/0032/23) |
| 4 | Classification | Emmanuel Rotich (COM/0043/23) |
| 5 | Model evaluation and selection | Elisha Eddie |
| 6 | Decision trees and ensembles | Emmanuel Rotich |
| 7 | Unsupervised and reinforcement learning | Dariol Temko |
| 8 | Neural networks | Raphael Wanyama |
| 9.1 | Model packaging and web app | Abdul Razak |
| 9.2 | Responsible AI and model card | Whole group |
| 9.3 | Publishing | Whole group |

## How to run the notebooks
1. Open the Google Drive folder CSC4110_G01 and add a shortcut to My Drive, so that /content/drive/MyDrive/CSC4110_G01 exists in Colab.
2. Open a notebook with Google Colaboratory, choose Runtime > Restart session and run all.
3. Run them in task order: each task reads files saved by the earlier ones (data/stage1_students.csv, data/clean_students.csv, models/G01_model.joblib).

## How to run the app
    pip install -r app/requirements.txt
    streamlit run app/app.py

## What the app does, for a school principal
The app is like a careful second opinion for the scholarship list. A teacher types in six things about a student: hours of study each week, attendance, the last exam score, and how motivated, supported at home and well-resourced the student is. The app answers with a percentage chance that the student will score 70 or more and says whether to invite them for an interview. It was built on a public dataset, not on your pupils, so treat it as a help for your own judgement and never as a reason to lower expectations for any student.

## Project folders
data, notebooks, models, outputs, app, tests, report. Run python tests/smoke_test.py to check that the saved model loads and predicts.
"""
(base / "README.md").write_text(readme)
print(readme[:600], "...")

# CSC 4110 Group G01: Student Performance Factors

Kibabii University, CSC 4110 Machine Learning. We built and tested two models for Elimu Bora Trust on the public Student Performance Factors dataset: one predicts a student's exam score, the other suggests which students to invite for a mentorship scholarship interview (score of 70 or more).

## Public app
**https://REPLACE-WITH-YOUR-APP-NAME.streamlit.app**  (replace after deploying on Streamlit Community Cloud)

## Who owned each task
| Task | Content | Owner |
|---|---|---|
| 1 | Set-up and data | Whole group |
| 2.1 | Data inspection and c ...


In [3]:
# Part (b): write tests/smoke_test.py (load the model, predict three test rows, stop with an error on failure)
smoke = r"""import sys
from pathlib import Path

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path(__file__).resolve().parent.parent
FEATURES = ["Hours_Studied", "Attendance", "Previous_Scores",
            "Motivation_Level", "Parental_Involvement", "Access_to_Resources"]

try:
    # 1. load the saved pipeline
    model = joblib.load(ROOT / "models" / "G01_model.joblib")
    # 2. rebuild the same test set and predict three rows
    df = pd.read_csv(ROOT / "data" / "stage1_students.csv")
    df["high_score"] = (df["Exam_Score"] >= 70).astype(int)
    _, X_test, _, y_test = train_test_split(df[FEATURES], df["high_score"], test_size=0.2,
                                            random_state=42, stratify=df["high_score"])
    sample = X_test.head(3)
    predictions = model.predict(sample)
    probabilities = model.predict_proba(sample)[:, 1]
except Exception as error:
    # 3. stop with an error message if loading or predicting fails
    sys.exit("SMOKE TEST FAILED: " + str(error))

for i, (p, prob) in enumerate(zip(predictions, probabilities)):
    print(f"Row {i}: predicted class {p}, probability of 70 or more {prob:.3f}, actual {y_test.iloc[i]}")
print("SMOKE TEST PASSED")
"""
(base / "tests" / "smoke_test.py").write_text(smoke)
print("Wrote tests/smoke_test.py")

Wrote tests/smoke_test.py


In [4]:
# Run the smoke test from the project folder, as the brief asks
%cd /content/drive/MyDrive/CSC4110_G01
!python tests/smoke_test.py

/content/drive/MyDrive/CSC4110_G01
Row 0: predicted class 0, probability of 70 or more 0.000, actual 0
Row 1: predicted class 1, probability of 70 or more 0.991, actual 1
Row 2: predicted class 0, probability of 70 or more 0.313, actual 0
SMOKE TEST PASSED


In [5]:
# Part (c): deployment checklist - files that must be in the GitHub repository before deploying
needed = ['README.md', 'app/app.py', 'app/requirements.txt', 'models/G01_model.joblib',
          'tests/smoke_test.py', 'data/stage1_students.csv', 'report/model_card.md']
for f in needed:
    print(('OK       ' if (base / f).exists() else 'MISSING  ') + f)
print("README still has the placeholder link:", 'REPLACE-WITH-YOUR-APP-NAME' in (base / 'README.md').read_text())

OK       README.md
OK       app/app.py
OK       app/requirements.txt
OK       models/G01_model.joblib
OK       tests/smoke_test.py
OK       data/stage1_students.csv
OK       report/model_card.md
README still has the placeholder link: True


In [6]:
# Self-check: README and smoke test files exist and the smoke test passes
import subprocess, sys
assert (base / 'README.md').exists() and (base / 'tests' / 'smoke_test.py').exists()
result = subprocess.run([sys.executable, str(base / 'tests' / 'smoke_test.py')], capture_output=True, text=True)
assert result.returncode == 0 and 'SMOKE TEST PASSED' in result.stdout, result.stderr
print("Task 9.3 checks passed")

Task 9.3 checks passed


### Written Answer: Running the App and Deployment Risk Mitigation

A new user can run the application directly in any web browser via our public Streamlit link (or locally by installing `app/requirements.txt` and running `streamlit run app/app.py`), entering a student's six core attributes and clicking **Predict** to see whether an interview is recommended at the `0.3` threshold. One major risk of putting the model online is **over-reliance or misinterpretation of predictions**—such as staff treating a low score as a permanent label that lowers expectations for a student, or feeding out-of-range inputs into a model trained on a single external dataset. We reduce this risk by presenting predictions strictly as advisory screening prompts rather than final decisions, enforcing input validation bounds alongside automated smoke tests (`tests/smoke_test.py`) and pinned dependencies (`models/G01_model.joblib`), and requiring human review and ongoing subgroup monitoring in line with our model card safeguards.